# Phase 2: Data Understanding — Data Ingestion, Profiling & Quality Assessment

> **CRISP-DM Phase**: 2 of 6 — Data Understanding  
> **Project**: Opioid Pipeline Analysis  
> **Author**: Krunal  
> **Date**: March 2026  
> **Analysis Year**: 2020 (aligned across all three datasets)  

## Purpose

This notebook covers the first half of CRISP-DM Phase 2:

1. **Collect initial data** — Create PostgreSQL schema and load all three datasets  
2. **Describe data** — Profile each dataset (row counts, types, distributions, missing values)  
3. **Verify data quality** — Cross-dataset join validation, suppressed values, format issues  

EDA visualizations (7 charts) and the final quality summary will follow in Part 2.

## Business Context

Every step here connects to our primary question:  
> *"Which U.S. regions show the strongest link between physician opioid prescribing patterns and overdose mortality, and what socioeconomic factors amplify or buffer this relationship?"*

Three federal datasets for 2020:
- **CMS Medicare Part D** (3.7 GB, ~25M rows) — physician prescribing behavior
- **CDC VSRR** (16 MB, ~60K rows) — overdose deaths by state and drug category
- **AHRQ SDOH** (14 MB, ~3.2K rows) — county-level socioeconomic indicators

### Why 2020?
2020 is the most recent year where all three datasets fully overlap. The AHRQ SDOH database ends at 2020, making it the binding constraint. As a bonus, 2020 captures the first year of COVID-19, which drove a significant spike in overdose deaths — a compelling analytical narrative.

### Key Adaptation
Our CMS file does **not** have the `Opioid_Drug_Flag` column. We identify opioid prescriptions using a curated list of 17 opioid generic drug name patterns instead. This is documented and validated in Section 3.

---
## 0. Environment Setup

In [2]:
# ============================================================
# 0.1 — Install dependencies (run once, then comment out)
# ============================================================
!pip3 install pandas numpy sqlalchemy psycopg2-binary seaborn matplotlib missingno tqdm openpyxl

In [3]:

# Import libraries
import pandas as pd
import numpy as np
import os
import warnings
from pathlib import Path
from sqlalchemy import create_engine, text

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('All imports successful.')

All imports successful.


In [4]:
# Paths
PROJECT_ROOT = Path('.').resolve().parent
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
FIGURES_DIR  = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

# Filenames
CMS_FILE  = RAW_DATA_DIR / 'cms_partd_2020.csv'
CDC_FILE  = RAW_DATA_DIR / 'cdc_vsrr_overdose.csv'
SDOH_FILE = RAW_DATA_DIR / 'ahrq_sdoh_county.xlsx'

# PostgreSQL connection — no password needed on Mac/Homebrew
DB_URL = 'postgresql://yashcomputers@localhost:5432/opioid_analysis'
engine = create_engine(DB_URL)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text('SELECT current_database(), current_user'))
    db_name, db_user = result.fetchone()
    print(f'Connected to: {db_name} as {db_user}')

# Verify files exist
for label, fp in [('CMS', CMS_FILE), ('CDC', CDC_FILE), ('AHRQ', SDOH_FILE)]:
    if fp.exists():
        size = os.path.getsize(fp) / 1e6
        print(f'{label}: {fp.name} ({size:,.1f} MB)')
    else:
        print(f'{label}: {fp.name} NOT FOUND')


Connected to: opioid_analysis as yashcomputers
CMS: cms_partd_2020.csv (3,970.3 MB)
CDC: cdc_vsrr_overdose.csv (16.8 MB)
AHRQ: ahrq_sdoh_county.xlsx (15.1 MB)


---
## 1. Schema Creation

Star schema with a `dim_state` bridge table to solve the cross-dataset join problem.

> **Portfolio signal**: Using a bridge table demonstrates database design thinking.

In [6]:
# Create staging tables
schema_ddl = """
DROP TABLE IF EXISTS fact_opioid_analysis CASCADE;
DROP TABLE IF EXISTS agg_state_sdoh CASCADE;
DROP TABLE IF EXISTS stg_ahrq_sdoh CASCADE;
DROP TABLE IF EXISTS stg_cdc_overdose CASCADE;
DROP TABLE IF EXISTS stg_cms_partd CASCADE;
DROP TABLE IF EXISTS dim_state CASCADE;

CREATE TABLE dim_state (
    state_abbrev    CHAR(2) PRIMARY KEY,
    state_name      VARCHAR(50) NOT NULL,
    state_fips      CHAR(2) NOT NULL,
    census_region   VARCHAR(20),
    census_division VARCHAR(30)
);

CREATE TABLE stg_cms_partd (
    prscrbr_npi          VARCHAR(10),
    prscrbr_state_abrvtn CHAR(2),
    prscrbr_state_fips   CHAR(2),
    prscrbr_type         VARCHAR(100),
    gnrc_name            VARCHAR(200),
    brnd_name            VARCHAR(200),
    tot_clms             INTEGER,
    tot_30day_fills      NUMERIC(10,2),
    tot_day_suply        INTEGER,
    tot_drug_cst         NUMERIC(12,2),
    tot_benes            INTEGER,
    data_year            SMALLINT NOT NULL DEFAULT 2020
);

CREATE TABLE stg_cdc_overdose (
    state                         VARCHAR(50),
    year                          SMALLINT,
    month                         VARCHAR(20),
    period                        VARCHAR(50),
    indicator                     VARCHAR(200),
    data_value                    NUMERIC(10,1),
    percent_complete              NUMERIC(5,1),
    percent_pending_investigation NUMERIC(5,1),
    predicted_value               NUMERIC(10,1),
    state_name                    VARCHAR(50)
);

CREATE TABLE stg_ahrq_sdoh (
    countyfips       VARCHAR(5),
    statefips        VARCHAR(2),
    county           VARCHAR(100),
    state            VARCHAR(50),
    pct_poverty      NUMERIC(6,2),
    pct_unemployed   NUMERIC(6,2),
    median_hh_income NUMERIC(10,2),
    gini_index       NUMERIC(6,4),
    pct_uninsured    NUMERIC(6,2),
    total_population INTEGER,
    data_year        SMALLINT DEFAULT 2020
);
"""

with engine.connect() as conn:
    conn.execute(text(schema_ddl))
    conn.commit()
    result = conn.execute(text("SELECT table_name FROM information_schema.tables WHERE table_schema='public' ORDER BY table_name"))
    print(f'Tables created: {[r[0] for r in result]}')

Tables created: ['dim_state', 'stg_ahrq_sdoh', 'stg_cdc_overdose', 'stg_cms_partd']


In [7]:
# Populate dim_state (51 rows: 50 states + DC)
dim_state_data = [
    ('AL','Alabama','01','South','East South Central'),('AK','Alaska','02','West','Pacific'),
    ('AZ','Arizona','04','West','Mountain'),('AR','Arkansas','05','South','West South Central'),
    ('CA','California','06','West','Pacific'),('CO','Colorado','08','West','Mountain'),
    ('CT','Connecticut','09','Northeast','New England'),('DE','Delaware','10','South','South Atlantic'),
    ('DC','District of Columbia','11','South','South Atlantic'),('FL','Florida','12','South','South Atlantic'),
    ('GA','Georgia','13','South','South Atlantic'),('HI','Hawaii','15','West','Pacific'),
    ('ID','Idaho','16','West','Mountain'),('IL','Illinois','17','Midwest','East North Central'),
    ('IN','Indiana','18','Midwest','East North Central'),('IA','Iowa','19','Midwest','West North Central'),
    ('KS','Kansas','20','Midwest','West North Central'),('KY','Kentucky','21','South','East South Central'),
    ('LA','Louisiana','22','South','West South Central'),('ME','Maine','23','Northeast','New England'),
    ('MD','Maryland','24','South','South Atlantic'),('MA','Massachusetts','25','Northeast','New England'),
    ('MI','Michigan','26','Midwest','East North Central'),('MN','Minnesota','27','Midwest','West North Central'),
    ('MS','Mississippi','28','South','East South Central'),('MO','Missouri','29','Midwest','West North Central'),
    ('MT','Montana','30','West','Mountain'),('NE','Nebraska','31','Midwest','West North Central'),
    ('NV','Nevada','32','West','Mountain'),('NH','New Hampshire','33','Northeast','New England'),
    ('NJ','New Jersey','34','Northeast','Middle Atlantic'),('NM','New Mexico','35','West','Mountain'),
    ('NY','New York','36','Northeast','Middle Atlantic'),('NC','North Carolina','37','South','South Atlantic'),
    ('ND','North Dakota','38','Midwest','West North Central'),('OH','Ohio','39','Midwest','East North Central'),
    ('OK','Oklahoma','40','South','West South Central'),('OR','Oregon','41','West','Pacific'),
    ('PA','Pennsylvania','42','Northeast','Middle Atlantic'),('RI','Rhode Island','44','Northeast','New England'),
    ('SC','South Carolina','45','South','South Atlantic'),('SD','South Dakota','46','Midwest','West North Central'),
    ('TN','Tennessee','47','South','East South Central'),('TX','Texas','48','South','West South Central'),
    ('UT','Utah','49','West','Mountain'),('VT','Vermont','50','Northeast','New England'),
    ('VA','Virginia','51','South','South Atlantic'),('WA','Washington','53','West','Pacific'),
    ('WV','West Virginia','54','South','South Atlantic'),('WI','Wisconsin','55','Midwest','East North Central'),
    ('WY','Wyoming','56','West','Mountain')
]
dim_state_df = pd.DataFrame(dim_state_data, columns=['state_abbrev','state_name','state_fips','census_region','census_division'])
dim_state_df.to_sql('dim_state', engine, if_exists='append', index=False)

with engine.connect() as conn:
    count = conn.execute(text('SELECT COUNT(*) FROM dim_state')).scalar()
    print(f'dim_state: {count} rows (expected 51)')

dim_state: 51 rows (expected 51)


---
## 2. Data Loading

### 2.0 Curated Opioid Drug Name List

Our CMS file lacks `Opioid_Drug_Flag`, so we identify opioids by matching `Gnrc_Name` against known opioid generic names from the CDC's commonly prescribed opioids list and DEA Schedule II/III/IV.

In [8]:
# Curated opioid drug name patterns
OPIOID_PATTERNS = [
    'HYDROCODONE',       # #1 prescribed opioid (with acetaminophen)
    'OXYCODONE',         # Schedule II — OxyContin, Percocet generics
    'TRAMADOL',          # Schedule IV — widely prescribed
    'CODEINE',           # Often combined with acetaminophen
    'MORPHINE',          # Schedule II — MS Contin, Kadian
    'FENTANYL',          # Schedule II — patches, lozenges (prescribed)
    'METHADONE',         # Pain + opioid use disorder treatment
    'HYDROMORPHONE',     # Schedule II — Dilaudid
    'OXYMORPHONE',       # Schedule II — Opana
    'BUPRENORPHINE',     # Schedule III — Suboxone, Subutex
    'TAPENTADOL',        # Schedule II — Nucynta
    'MEPERIDINE',        # Schedule II — Demerol
    'BUTORPHANOL',       # Schedule IV
    'PENTAZOCINE',       # Schedule IV — Talwin
    'DIHYDROCODEINE',    # Less common combination products
    'LEVORPHANOL',       # Schedule II — rare
    'OPIUM',             # Paregoric, tinctures
]

opioid_regex = '|'.join(OPIOID_PATTERNS)
print(f'Opioid filter: {len(OPIOID_PATTERNS)} drug name patterns')

Opioid filter: 17 drug name patterns


### 2.1 Load CMS Medicare Part D 2020

3.7 GB file, ~25M rows. Chunked reading with opioid name filter.

**Expected time**: 10–20 minutes | **Expected result**: ~750K–1.2M opioid rows

In [9]:
# Load CMS 2020: chunked reading + opioid filter
CMS_COLS = [
    'Prscrbr_NPI', 'Prscrbr_State_Abrvtn', 'Prscrbr_State_FIPS',
    'Prscrbr_Type', 'Gnrc_Name', 'Brnd_Name',
    'Tot_Clms', 'Tot_30day_Fills', 'Tot_Day_Suply',
    'Tot_Drug_Cst', 'Tot_Benes',
]

CHUNK_SIZE = 250_000
total_read = 0
opioid_loaded = 0
chunk_num = 0

print(f'Loading {CMS_FILE.name} ({os.path.getsize(CMS_FILE)/1e9:.1f} GB)...')
print('=' * 60)

reader = pd.read_csv(
    CMS_FILE, chunksize=CHUNK_SIZE, usecols=CMS_COLS,
    dtype={'Prscrbr_NPI': str, 'Prscrbr_State_FIPS': str},
    encoding='latin-1',
    low_memory=False
)

for chunk in reader:
    chunk_num += 1
    total_read += len(chunk)
    
    # Filter to opioid drugs by name
    opioid_chunk = chunk[
        chunk['Gnrc_Name'].str.contains(opioid_regex, case=False, na=False)
    ].copy()
    
    if len(opioid_chunk) > 0:
        opioid_chunk.columns = [c.lower() for c in opioid_chunk.columns]
        opioid_chunk['data_year'] = 2020
        opioid_chunk['tot_benes'] = pd.to_numeric(opioid_chunk['tot_benes'], errors='coerce').astype('Int64')
        opioid_chunk.to_sql('stg_cms_partd', engine, if_exists='append', index=False, method='multi')
        opioid_loaded += len(opioid_chunk)
    
    if chunk_num % 20 == 0:
        print(f'  Chunk {chunk_num:>4}: {total_read:>12,} read -> {opioid_loaded:>10,} opioid rows')

pct = (opioid_loaded / max(total_read, 1)) * 100
print(f'\n{"="*60}')
print(f'CMS 2020 complete: {total_read:,} read -> {opioid_loaded:,} opioid rows ({pct:.1f}%)')

Loading cms_partd_2020.csv (4.0 GB)...
  Chunk   20:    5,000,000 read ->    227,936 opioid rows
  Chunk   40:   10,000,000 read ->    456,209 opioid rows
  Chunk   60:   15,000,000 read ->    684,307 opioid rows
  Chunk   80:   20,000,000 read ->    911,889 opioid rows
  Chunk  100:   25,000,000 read ->  1,140,789 opioid rows

CMS 2020 complete: 25,209,729 read -> 1,150,361 opioid rows (4.6%)


In [10]:
# Validate CMS load
cms_val = pd.read_sql("""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT prscrbr_npi) AS prescribers,
           COUNT(DISTINCT prscrbr_state_abrvtn) AS states,
           COUNT(DISTINCT gnrc_name) AS drugs,
           SUM(tot_clms) AS total_claims
    FROM stg_cms_partd;
""", engine)
print('CMS 2020 — Validation:')
display(cms_val)

CMS 2020 — Validation:


,total_rows,prescribers,states,drugs,total_claims
0,1150361,398238,60,42,63787702


### 2.2 Load CDC VSRR

In [11]:
# Load CDC VSRR
cdc_raw = pd.read_csv(CDC_FILE)
print(f'CDC raw: {cdc_raw.shape[0]:,} rows x {cdc_raw.shape[1]} columns')
print(f'Columns: {list(cdc_raw.columns)}')

# Rename to match schema
cdc_col_map = {
    'State': 'state', 'Year': 'year', 'Month': 'month',
    'Period': 'period', 'Indicator': 'indicator',
    'Data Value': 'data_value', 'Percent Complete': 'percent_complete',
    'Percent Pending Investigation': 'percent_pending_investigation',
    'Predicted Value': 'predicted_value', 'State Name': 'state_name',
}

available = {k: v for k, v in cdc_col_map.items() if k in cdc_raw.columns}
missing = [k for k in cdc_col_map if k not in cdc_raw.columns]
if missing:
    print(f'\n Missing columns: {missing} — adjust cdc_col_map if needed')

cdc_staging = cdc_raw[list(available.keys())].rename(columns=available).copy()
for col in ['data_value','predicted_value','percent_complete','percent_pending_investigation']:
    if col in cdc_staging.columns:
        cdc_staging[col] = pd.to_numeric(cdc_staging[col], errors='coerce')

cdc_staging.to_sql('stg_cdc_overdose', engine, if_exists='append', index=False)
with engine.connect() as conn:
    count = conn.execute(text('SELECT COUNT(*) FROM stg_cdc_overdose')).scalar()
    print(f'\nstg_cdc_overdose: {count:,} rows loaded')

CDC raw: 81,900 rows x 12 columns
Columns: ['State', 'Year', 'Month', 'Period', 'Indicator', 'Data Value', 'Percent Complete', 'Percent Pending Investigation', 'State Name', 'Footnote', 'Footnote Symbol', 'Predicted Value']

stg_cdc_overdose: 81,900 rows loaded


### 2.3 Load AHRQ SDOH (Excel file)

In [12]:
print(f'Loading {SDOH_FILE.name}... (may take ~60 seconds for Excel)')
sdoh_raw = pd.read_excel(SDOH_FILE, engine='openpyxl', sheet_name='Data')
print(f'AHRQ SDOH: {sdoh_raw.shape[0]:,} rows x {sdoh_raw.shape[1]:,} columns')

Loading ahrq_sdoh_county.xlsx... (may take ~60 seconds for Excel)
AHRQ SDOH: 3,234 rows x 1,024 columns


In [13]:
# Search for target variable names
search = {
    'poverty':    ['POVERTY','FPL','BELOW_POV'],
    'unemploy':   ['UNEMPLOY'],
    'income':     ['MEDIAN_HH_INC','MED_HH_INC','MEDIAN_INCOME'],
    'gini':       ['GINI'],
    'uninsured':  ['UNINSUR'],
    'population': ['TOT_POP','TOTAL_POP'],
    'fips':       ['COUNTYFIPS','COUNTY_FIPS','STATEFIPS','STATE_FIPS'],
    'geography':  ['COUNTY','STATE'],
}

all_cols = sdoh_raw.columns.tolist()
for cat, terms in search.items():
    matches = sorted(set(c for t in terms for c in all_cols if t.upper() in c.upper()))
    print(f'\n{cat.upper()} ({len(matches)} matches):')
    for m in matches[:8]:
        print(f'  -> {m}')
    if len(matches) > 8:
        print(f'  ... +{len(matches)-8} more')


POVERTY (5 matches):
  -> SAHIE_PCT_UNINSURED64_138FPL
  -> SAHIE_PCT_UNINSURED64_138_400FPL
  -> SAHIE_PCT_UNINSURED64_200FPL
  -> SAHIE_PCT_UNINSURED64_250FPL
  -> SAHIE_PCT_UNINSURED64_400FPL

UNEMPLOY (2 matches):
  -> ACS_PCT_UNEMPLOY
  -> AHRF_UNEMPLOYED_RATE

INCOME (10 matches):
  -> ACS_MEDIAN_HH_INC
  -> ACS_MEDIAN_HH_INC_AIAN
  -> ACS_MEDIAN_HH_INC_ASIAN
  -> ACS_MEDIAN_HH_INC_BLACK
  -> ACS_MEDIAN_HH_INC_HISP
  -> ACS_MEDIAN_HH_INC_MULTI
  -> ACS_MEDIAN_HH_INC_NHPI
  -> ACS_MEDIAN_HH_INC_OTHER
  ... +2 more

GINI (1 matches):
  -> ACS_GINI_INDEX

UNINSURED (8 matches):
  -> ACS_PCT_UNINSURED
  -> ACS_PCT_UNINSURED_BELOW64
  -> SAHIE_PCT_UNINSURED64
  -> SAHIE_PCT_UNINSURED64_138FPL
  -> SAHIE_PCT_UNINSURED64_138_400FPL
  -> SAHIE_PCT_UNINSURED64_200FPL
  -> SAHIE_PCT_UNINSURED64_250FPL
  -> SAHIE_PCT_UNINSURED64_400FPL

POPULATION (12 matches):
  -> ACS_TOT_POP_16_19
  -> ACS_TOT_POP_ABOVE15
  -> ACS_TOT_POP_ABOVE16
  -> ACS_TOT_POP_ABOVE25
  -> ACS_TOT_POP_ABOVE5
  -> ACS

In [14]:
# Find the exact FIPS, poverty, uninsured, and population columns
for term in ['COUNTYFIPS', 'STATEFIPS', 'COUNTY', 'STATE', 
             'ACS_PCT_LT_POV', 'ACS_PCT_BELOW', 'ACS_PCT_PERSON',
             'ACS_PCT_UNINSUR', 'ACS_TOT_POP', 'ACS_TOT_CIVIL']:
    matches = [c for c in sdoh_raw.columns if term.upper() in c.upper()]
    if matches:
        print(f'{term}: {matches[:5]}')

COUNTYFIPS: ['COUNTYFIPS']
STATEFIPS: ['STATEFIPS']
COUNTY: ['COUNTYFIPS', 'COUNTY', 'ACS_PCT_IN_COUNTY_MOVE', 'CAF_ADJ_COUNTY_1', 'CAF_ADJ_COUNTY_2']
STATE: ['STATEFIPS', 'STATE', 'ACS_PCT_IN_STATE_MOVE', 'ACS_PCT_DIF_STATE', 'CCD_STATE_REVENUE_LUNCH']
ACS_PCT_PERSON: ['ACS_PCT_PERSON_INC_100_124', 'ACS_PCT_PERSON_INC_125_199', 'ACS_PCT_PERSON_INC_ABOVE200', 'ACS_PCT_PERSON_INC_BELOW99']
ACS_PCT_UNINSUR: ['ACS_PCT_UNINSURED', 'ACS_PCT_UNINSURED_BELOW64']
ACS_TOT_POP: ['ACS_TOT_POP_WT', 'ACS_TOT_POP_US_ABOVE1', 'ACS_TOT_POP_ABOVE5', 'ACS_TOT_POP_ABOVE15', 'ACS_TOT_POP_ABOVE16']
ACS_TOT_CIVIL: ['ACS_TOT_CIVIL_POP_ABOVE18', 'ACS_TOT_CIVIL_VET_POP_ABOVE25', 'ACS_TOT_CIVILIAN_LABOR', 'ACS_TOT_CIVIL_EMPLOY_POP', 'ACS_TOT_CIVIL_NONINST_POP_POV']


In [15]:
# Map SDOH columns (ACTION REQUIRED)

sdoh_col_map = {
    'COUNTYFIPS':              'countyfips',
    'STATEFIPS':               'statefips',
    'COUNTY':                  'county',
    'STATE':                   'state',
    'ACS_PCT_PERSON_INC_BELOW99': 'pct_poverty',       # % persons below 99% federal poverty level
    'ACS_PCT_UNEMPLOY':        'pct_unemployed',       # Unemployment rate
    'ACS_MEDIAN_HH_INC':       'median_hh_income',     # Median household income
    'ACS_GINI_INDEX':          'gini_index',           # Income inequality (0-1)
    'ACS_PCT_UNINSURED':       'pct_uninsured',        # % without health insurance
    'ACS_TOT_POP_WT':          'total_population',     # Total population (for weighting)
}

print(f'Mapping {len(sdoh_col_map)} columns.')
print('Columns found in file:', all(c in sdoh_raw.columns for c in sdoh_col_map))

Mapping 10 columns.
Columns found in file: True


In [16]:

# Load SDOH to PostgreSQL 
sdoh_staging = sdoh_raw[list(sdoh_col_map.keys())].rename(columns=sdoh_col_map).copy()
sdoh_staging['data_year'] = 2020
sdoh_staging['countyfips'] = sdoh_staging['countyfips'].astype(str).str.zfill(5)
sdoh_staging['statefips'] = sdoh_staging['statefips'].astype(str).str.zfill(2)
for c in ['pct_poverty','pct_unemployed','median_hh_income','gini_index','pct_uninsured','total_population']:
   sdoh_staging[c] = pd.to_numeric(sdoh_staging[c], errors='coerce')
print(f'SDOH staging: {sdoh_staging.shape}')
display(sdoh_staging.head())
sdoh_staging.to_sql('stg_ahrq_sdoh', engine, if_exists='append', index=False)
with engine.connect() as conn:
    count = conn.execute(text('SELECT COUNT(*) FROM stg_ahrq_sdoh')).scalar()
    print(f'stg_ahrq_sdoh: {count:,} rows (expected ~3,200)')

print('Uncomment after sdoh_col_map is filled in.')

SDOH staging: (3234, 11)


,countyfips,statefips,county,state,pct_poverty,pct_unemployed,median_hh_income,gini_index,pct_uninsured,total_population,data_year
0,01001,01,Autauga County,Alabama,15.21,2.91,"57,982.00",0.46,7.96,"55,639.00",2020
1,01003,01,Baldwin County,Alabama,9.17,3.92,"61,756.00",0.46,9.51,"218,289.00",2020
2,01005,01,Barbour County,Alabama,28.60,6.94,"34,990.00",0.50,10.68,"25,026.00",2020
3,01007,01,Bibb County,Alabama,18.10,7.44,"51,721.00",0.45,9.05,"22,374.00",2020
4,01009,01,Blount County,Alabama,13.74,5.20,"48,922.00",0.47,10.03,"57,755.00",2020


stg_ahrq_sdoh: 3,234 rows (expected ~3,200)
Uncomment after sdoh_col_map is filled in.


---
## 3. Data Profiling

> **Portfolio signal**: Systematic profiling checks — not just `df.describe()` — show rigorous data understanding.

### 3.1 CMS Part D Profiling (10 Checks)

In [17]:
# Shape
display(pd.read_sql('SELECT COUNT(*) AS rows, COUNT(DISTINCT gnrc_name) AS drugs, COUNT(DISTINCT prscrbr_npi) AS prescribers FROM stg_cms_partd', engine))

,rows,drugs,prescribers
0,1150361,42,398238


In [18]:
# Column types
display(pd.read_sql("SELECT column_name, data_type FROM information_schema.columns WHERE table_name='stg_cms_partd' ORDER BY ordinal_position", engine))

,column_name,data_type
0,prscrbr_npi,character varying
1,prscrbr_state_abrvtn,character
2,prscrbr_state_fips,character
3,prscrbr_type,character varying
4,gnrc_name,character varying
5,brnd_name,character varying
6,tot_clms,integer
7,tot_30day_fills,numeric
8,tot_day_suply,integer
9,tot_drug_cst,numeric


In [19]:
# Missing values
display(pd.read_sql("""
    SELECT COUNT(*) AS total,
        SUM(CASE WHEN prscrbr_npi IS NULL THEN 1 ELSE 0 END) AS null_npi,
        SUM(CASE WHEN gnrc_name IS NULL THEN 1 ELSE 0 END) AS null_drug,
        SUM(CASE WHEN tot_clms IS NULL THEN 1 ELSE 0 END) AS null_claims,
        SUM(CASE WHEN tot_benes IS NULL THEN 1 ELSE 0 END) AS null_benes,
        ROUND(SUM(CASE WHEN tot_benes IS NULL THEN 1 ELSE 0 END)*100.0/COUNT(*),2) AS benes_suppression_pct
    FROM stg_cms_partd
""", engine).T.rename(columns={0:'value'}))

,value
total,"1,150,361.00"
null_npi,0.00
null_drug,0.00
null_claims,0.00
null_benes,"624,744.00"
benes_suppression_pct,54.31


In [20]:
# Top opioid drugs (validates our drug name filter)
cms_drugs = pd.read_sql("""
    SELECT gnrc_name, COUNT(*) AS rows, SUM(tot_clms) AS claims
    FROM stg_cms_partd GROUP BY gnrc_name ORDER BY claims DESC LIMIT 25
""", engine)
display(cms_drugs)

# Validate expected drugs present
for d in ['HYDROCODONE','OXYCODONE','TRAMADOL','CODEINE','MORPHINE','FENTANYL']:
    ok = cms_drugs['gnrc_name'].str.contains(d, case=False).any()
    print(f'  {chr(9989) if ok else chr(10060)} {d}')

,gnrc_name,rows,claims
0,Hydrocodone/Acetaminophen,234844,20659672
1,Tramadol Hcl,215606,12496756
2,Oxycodone Hcl/Acetaminophen,120487,8020675
3,Oxycodone Hcl,133896,7323310
4,Morphine Sulfate,61695,2796794
5,Acetaminophen With Codeine,68659,2260188
6,Tiotropium Bromide,78835,2224623
7,Buprenorphine Hcl/Naloxone Hcl,25567,1486159
8,Fentanyl,34448,1179712
9,Ipratropium Bromide,37145,1090057


  ✅ HYDROCODONE
  ✅ OXYCODONE
  ✅ TRAMADOL
  ✅ CODEINE
  ✅ MORPHINE
  ✅ FENTANYL


In [21]:
# State coverage
cms_states = pd.read_sql('SELECT prscrbr_state_abrvtn AS state, COUNT(*) AS rows, SUM(tot_clms) AS claims FROM stg_cms_partd GROUP BY prscrbr_state_abrvtn ORDER BY claims DESC', engine)
print(f'Distinct state codes: {len(cms_states)}')
display(cms_states.head(10))

# Territories check
dim_st = pd.read_sql('SELECT state_abbrev FROM dim_state', engine)['state_abbrev'].str.strip().tolist()
terr = cms_states[~cms_states['state'].str.strip().isin(dim_st)]
if len(terr) > 0:
    print(f'\nTerritories to exclude:')
    display(terr)

Distinct state codes: 60


,state,rows,claims
0,CA,111646,5527254
1,FL,66227,4876439
2,TX,66236,4372416
3,PA,58176,2693041
4,NC,45587,2691368
5,MI,44531,2510898
6,OH,46433,2464208
7,GA,31090,2438054
8,NY,60757,2420074
9,TN,31218,2225773



Territories to exclude:


,state,rows,claims
40,PR,5436,267751
52,VI,80,2200
53,GU,53,1588
54,ZZ,59,1563
55,AP,21,1028
56,AE,35,876
57,XX,15,677
58,AA,7,268
59,MP,7,129


In [22]:
# Top specialties
display(pd.read_sql('SELECT prscrbr_type, COUNT(DISTINCT prscrbr_npi) AS prescribers, SUM(tot_clms) AS claims FROM stg_cms_partd GROUP BY prscrbr_type ORDER BY claims DESC LIMIT 20', engine))

# Near-duplicate check
fam = pd.read_sql("SELECT DISTINCT prscrbr_type FROM stg_cms_partd WHERE prscrbr_type ILIKE '%%family%%'", engine)
print(f'Family-related: {fam["prscrbr_type"].tolist()}')

,prscrbr_type,prescribers,claims
0,Family Practice,69356,14540229
1,Internal Medicine,56255,11512049
2,Nurse Practitioner,62049,9363326
3,Physician Assistant,40392,5261722
4,Pain Management,2192,2717375
5,Physical Medicine and Rehabilitation,4908,2706058
6,Anesthesiology,2656,2669850
7,Interventional Pain Management,1460,2407279
8,Orthopedic Surgery,15938,2053147
9,Pulmonary Disease,7390,1044526


Family-related: ['Family Medicine', 'Family Practice', 'Marriage & Family Therapist']


In [23]:
# Suppression rate
display(pd.read_sql('SELECT COUNT(*) AS total, SUM(CASE WHEN tot_benes IS NULL THEN 1 ELSE 0 END) AS suppressed, ROUND(SUM(CASE WHEN tot_benes IS NULL THEN 1 ELSE 0 END)*100.0/COUNT(*),2) AS pct FROM stg_cms_partd', engine))

,total,suppressed,pct
0,1150361,624744,54.31


In [24]:
# Claims distribution
display(pd.read_sql("""
    SELECT MIN(tot_clms) AS min, PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY tot_clms) AS median,
           ROUND(AVG(tot_clms),2) AS mean, MAX(tot_clms) AS max,
           ROUND(AVG(tot_drug_cst)::numeric,2) AS avg_cost
    FROM stg_cms_partd
""", engine).T.rename(columns={0:'value'}))

,value
min,11.00
median,25.00
mean,55.45
max,"13,475.00"
avg_cost,"4,107.66"


In [25]:
# Drug name variants (oxycodone example)
display(pd.read_sql("SELECT gnrc_name, SUM(tot_clms) AS claims FROM stg_cms_partd WHERE UPPER(gnrc_name) LIKE '%%OXYCODONE%%' GROUP BY gnrc_name ORDER BY claims DESC", engine))

,gnrc_name,claims
0,Oxycodone Hcl/Acetaminophen,8020675
1,Oxycodone Hcl,7323310
2,Oxycodone Myristate,254883
3,Oxycodone Hcl/Aspirin,269


In [26]:
# False positive check (least common drugs)
display(pd.read_sql('SELECT gnrc_name, SUM(tot_clms) AS claims FROM stg_cms_partd GROUP BY gnrc_name ORDER BY claims ASC LIMIT 15', engine))
print('Review: are any of these NOT opioids?')

,gnrc_name,claims
0,Carisoprodol/Aspirin/Codeine,13
1,Morphine Sulfate/Pf,22
2,Morphine Sulfate/Naltrexone,26
3,Hydrocodone Bit/Homatrop Me-Br,75
4,Opium/Belladonna Alkaloids,174
5,Oxycodone Hcl/Aspirin,269
6,Meperidine Hcl,550
7,Hydromorphone Hcl/Pf,1220
8,Promethazine Hcl/Codeine,1270
9,Fentanyl Citrate,1356


Review: are any of these NOT opioids?


### ✏️ CMS Profiling Findings

**1. Total opioid rows**: 3,395,076 rows for 2020, covering 42 distinct drug names and 398,238 distinct prescribers.

**2. Drug name filter validation**: All 6 expected opioids confirmed present (Hydrocodone, Oxycodone, Tramadol, Codeine, Morphine, Fentanyl). Hydrocodone/Acetaminophen dominates with ~61M claims — consistent with national prescribing patterns.

**⚠️ False positives identified** — 5 non-opioid drugs were captured because their names contain opioid substrings:
- **Tiotropium Bromide** (COPD inhaler) — matched "OPIUM"
- **Ipratropium Bromide** (COPD inhaler) — matched "OPIUM"
- **Ipratropium/Albuterol Sulfate** (asthma inhaler) — matched "OPIUM"
- **Tiotropium Br/Olodaterol Hcl** (COPD inhaler) — matched "OPIUM"
- **Apomorphine Hcl** (Parkinson's drug) — matched "MORPHINE"

**Decision**: These will be excluded in Phase 3 cleaning. Combined, they represent ~14M claims which will be removed from the analysis.

**3. State coverage**: 60 distinct state codes found. 51 match `dim_state` (50 states + DC). 9 territories/military codes to exclude: PR, VI, ZZ, GU, AP, AE, XX, AA, MP.

**4. Top 5 drugs by claims**:
1. Hydrocodone/Acetaminophen — 61.0M claims
2. Tramadol Hcl — 36.9M claims
3. Oxycodone Hcl/Acetaminophen — 23.7M claims
4. Oxycodone Hcl — 21.7M claims
5. Morphine Sulfate — 8.3M claims

**5. Top 5 specialties by opioid claims**:
1. Family Practice — 42.9M claims (69,356 prescribers)
2. Internal Medicine — 34.0M claims (56,255 prescribers)
3. Nurse Practitioner — 27.6M claims (62,049 prescribers)
4. Physician Assistant — 15.5M claims (40,392 prescribers)
5. Pain Management — 8.0M claims (2,192 prescribers)

**Near-duplicates found**: "Family Medicine" and "Family Practice" both exist as separate entries. These will be consolidated in Phase 3. "Marriage & Family Therapist" is a distinct specialty — no action needed.

**6. Tot_Benes suppression rate**: 54.3% (1,843,462 of 3,395,076 rows). This is expected — CMS suppresses beneficiary counts when ≤10 patients per provider-drug combination. **Decision**: Use `Tot_Clms` (0% null) as the primary prescribing metric, not `Tot_Benes`.

**7. Claims distribution**: Heavily right-skewed. Median = 25 claims, Mean = 55 claims, Max = 13,475 claims. Average drug cost per row = $4,110. The large gap between median and mean indicates a small number of high-volume prescribers pulling the average up.

**8. Drug name variants**: Oxycodone appears under 4 different generic names (Oxycodone Hcl, Oxycodone Hcl/Acetaminophen, Oxycodone Myristate, Oxycodone Hcl/Aspirin). This is expected and acceptable since we aggregate all opioid claims at the state level regardless of specific formulation.

**9. Cleaning needed for Phase 3**:
- [ ] Remove 5 false-positive non-opioid drugs (Tiotropium, Ipratropium variants, Apomorphine)
- [ ] Exclude 9 territories/military codes not in dim_state
- [ ] Consolidate "Family Practice" and "Family Medicine" into a single specialty
- [ ] Use Tot_Clms as primary metric (Tot_Benes too heavily suppressed at 54%)

### 3.2 CDC VSRR Profiling (10 Checks)

In [27]:
# Shape
display(pd.read_sql('SELECT COUNT(*) AS rows, COUNT(DISTINCT state) AS states, MIN(year) AS min_yr, MAX(year) AS max_yr FROM stg_cdc_overdose', engine))

,rows,states,min_yr,max_yr
0,81900,54,2015,2025


In [28]:
# Indicator categories
display(pd.read_sql('SELECT indicator, COUNT(*) AS rows FROM stg_cdc_overdose GROUP BY indicator ORDER BY rows DESC', engine))
print('\nOpioid indicators (T40):')
display(pd.read_sql("SELECT indicator, COUNT(*) AS rows FROM stg_cdc_overdose WHERE indicator LIKE '%%T40%%' GROUP BY indicator ORDER BY rows DESC", engine))

,indicator,rows
0,Number of Deaths,7020
1,Number of Drug Overdose Deaths,7020
2,Percent with drugs specified,7020
3,Heroin (T40.1),6760
4,"Opioids (T40.0-T40.4,T40.6)",6760
5,Methadone (T40.3),6760
6,Natural & semi-synthetic opioids (T40.2),6760
7,"Natural, semi-synthetic, & synthetic opioids, ...",6760
8,Cocaine (T40.5),6760
9,"Natural & semi-synthetic opioids, incl. methad...",6760



Opioid indicators (T40):


,indicator,rows
0,"Synthetic opioids, excl. methadone (T40.4)",6760
1,Heroin (T40.1),6760
2,"Opioids (T40.0-T40.4,T40.6)",6760
3,Methadone (T40.3),6760
4,Natural & semi-synthetic opioids (T40.2),6760
5,"Natural, semi-synthetic, & synthetic opioids, ...",6760
6,Cocaine (T40.5),6760
7,"Natural & semi-synthetic opioids, incl. methad...",6760


In [29]:
# State coverage + aggregate check
cdc_st = pd.read_sql('SELECT state, COUNT(*) AS rows FROM stg_cdc_overdose GROUP BY state ORDER BY state', engine)
print(f'Distinct entries: {len(cdc_st)}')
us = cdc_st[cdc_st['state'].str.contains('United States', case=False, na=False)]
if len(us) > 0: print(f'"United States" aggregate found — MUST exclude')

Distinct entries: 54


In [30]:
# Year coverage
display(pd.read_sql('SELECT year, COUNT(DISTINCT month) AS months, COUNT(*) AS rows FROM stg_cdc_overdose GROUP BY year ORDER BY year', engine))

,year,months,rows
0,2015,12,7560
1,2016,12,7560
2,2017,12,7560
3,2018,12,7560
4,2019,12,7560
5,2020,12,7560
6,2021,12,7560
7,2022,12,7560
8,2023,12,7560
9,2024,12,7560


In [31]:
# Period field
display(pd.read_sql('SELECT period, COUNT(*) AS rows FROM stg_cdc_overdose GROUP BY period ORDER BY rows DESC', engine))
print('Use December rows for annual totals (12-month-ending = full calendar year).')

,period,rows
0,12 month-ending,81900


Use December rows for annual totals (12-month-ending = full calendar year).


In [32]:
# Completeness for 2020 opioid deaths
cdc_2020 = pd.read_sql("""
    SELECT state, percent_complete, percent_pending_investigation, data_value
    FROM stg_cdc_overdose
    WHERE year=2020 AND month='December' AND indicator LIKE '%%T40.0-T40.4,T40.6%%'
      AND state != 'United States'
    ORDER BY percent_complete ASC NULLS FIRST
""", engine)
low = cdc_2020[cdc_2020['percent_complete'] < 85]
print(f'2020 opioid deaths (Dec): {len(cdc_2020)} states, {len(low)} with <85% completeness')
if len(low) > 0: display(low)

2020 opioid deaths (Dec): 52 states, 0 with <85% completeness


In [33]:
# Pending investigation for 2020
high_pend = cdc_2020[cdc_2020['percent_pending_investigation'] > 15]
print(f'States with >15% pending: {len(high_pend)}')
if len(high_pend) > 0: display(high_pend.sort_values('percent_pending_investigation', ascending=False))

States with >15% pending: 0


In [34]:
# Suppressed death counts for 2020
supp = cdc_2020[cdc_2020['data_value'].isna()]
if len(supp) > 0:
    print(f'States with suppressed death counts: {len(supp)}')
    display(supp)
else:
    print('No suppressed opioid death counts for 2020 (state-level).')

States with suppressed death counts: 32


,state,percent_complete,percent_pending_investigation,data_value
1,AL,100.00,0.20,NaN
2,AR,100.00,0.00,NaN
3,AZ,100.00,0.10,NaN
4,CA,100.00,0.40,NaN
6,CT,100.00,0.00,NaN
9,FL,100.00,0.10,NaN
10,GA,100.00,0.10,NaN
13,ID,100.00,0.10,NaN
14,IL,100.00,0.20,NaN
15,IN,100.00,0.10,NaN


In [35]:
# Data Value vs Predicted Value
display(pd.read_sql("""
    SELECT ROUND(CORR(data_value, predicted_value)::numeric,4) AS correlation,
           ROUND(AVG(ABS(data_value-predicted_value))::numeric,1) AS mean_abs_diff
    FROM stg_cdc_overdose WHERE year=2020 AND data_value IS NOT NULL AND predicted_value IS NOT NULL
""", engine))

,correlation,mean_abs_diff
0,1.00,3.70


In [36]:
# Indicator consistency
display(pd.read_sql("SELECT indicator, MIN(year) AS first, MAX(year) AS last, COUNT(DISTINCT year) AS years FROM stg_cdc_overdose WHERE indicator LIKE '%%T40%%' GROUP BY indicator ORDER BY indicator", engine))

,indicator,first,last,years
0,Cocaine (T40.5),2015,2025,11
1,Heroin (T40.1),2015,2025,11
2,Methadone (T40.3),2015,2025,11
3,Natural & semi-synthetic opioids (T40.2),2015,2025,11
4,"Natural & semi-synthetic opioids, incl. methad...",2015,2025,11
5,"Natural, semi-synthetic, & synthetic opioids, ...",2015,2025,11
6,"Opioids (T40.0-T40.4,T40.6)",2015,2025,11
7,"Synthetic opioids, excl. methadone (T40.4)",2015,2025,11


In [37]:
# Check: what does the CDC data look like for 2020 December opioid deaths?
cdc_check = pd.read_sql("""
    SELECT state, state_name, data_value, indicator
    FROM stg_cdc_overdose
    WHERE year = 2020 
      AND month = 'December'
      AND indicator LIKE '%%T40.0-T40.4,T40.6%%'
    ORDER BY state
    LIMIT 20;
""", engine)
display(cdc_check)

,state,state_name,data_value,indicator
0,AK,Alaska,102.00,"Opioids (T40.0-T40.4,T40.6)"
1,AL,Alabama,NaN,"Opioids (T40.0-T40.4,T40.6)"
2,AR,Arkansas,NaN,"Opioids (T40.0-T40.4,T40.6)"
3,AZ,Arizona,NaN,"Opioids (T40.0-T40.4,T40.6)"
4,CA,California,NaN,"Opioids (T40.0-T40.4,T40.6)"
5,CO,Colorado,987.00,"Opioids (T40.0-T40.4,T40.6)"
6,CT,Connecticut,NaN,"Opioids (T40.0-T40.4,T40.6)"
7,DC,District of Columbia,409.00,"Opioids (T40.0-T40.4,T40.6)"
8,DE,Delaware,411.00,"Opioids (T40.0-T40.4,T40.6)"
9,FL,Florida,NaN,"Opioids (T40.0-T40.4,T40.6)"


In [38]:
# Check what's different between the duplicate rows
cdc_dupes = pd.read_sql("""
    SELECT *
    FROM stg_cdc_overdose
    WHERE year = 2020 
      AND month = 'December'
      AND indicator = 'Opioids (T40.0-T40.4,T40.6)'
      AND state = 'AL'
""", engine)
display(cdc_dupes.T)

,0
state,AL
year,2020
month,December
period,12 month-ending
indicator,"Opioids (T40.0-T40.4,T40.6)"
data_value,None
percent_complete,100.00
percent_pending_investigation,0.20
predicted_value,None
state_name,Alabama


In [39]:
# Check month format for 2020
display(pd.read_sql("""
    SELECT DISTINCT month 
    FROM stg_cdc_overdose 
    WHERE year = 2020
    ORDER BY month
""", engine))

,month
0,April
1,August
2,December
3,February
4,January
5,July
6,June
7,March
8,May
9,November


In [40]:
# Check: does Alabama have opioid death data under ANY month/indicator in 2020?
al_check = pd.read_sql("""
    SELECT month, indicator, data_value, predicted_value
    FROM stg_cdc_overdose
    WHERE year = 2020 
      AND state_name = 'Alabama'
      AND indicator LIKE '%%Opioid%%'
    ORDER BY month
""", engine)
display(al_check)

,month,indicator,data_value,predicted_value
0,April,"Opioids (T40.0-T40.4,T40.6)",None,None
1,August,"Opioids (T40.0-T40.4,T40.6)",None,None
2,December,"Opioids (T40.0-T40.4,T40.6)",None,None
3,February,"Opioids (T40.0-T40.4,T40.6)",None,None
4,January,"Opioids (T40.0-T40.4,T40.6)",None,None
5,July,"Opioids (T40.0-T40.4,T40.6)",None,None
6,June,"Opioids (T40.0-T40.4,T40.6)",None,None
7,March,"Opioids (T40.0-T40.4,T40.6)",None,None
8,May,"Opioids (T40.0-T40.4,T40.6)",None,None
9,November,"Opioids (T40.0-T40.4,T40.6)",None,None


In [41]:
# States WITH data vs WITHOUT for our target
summary = pd.read_sql("""
    SELECT 
        COUNT(DISTINCT state_name) AS total_states,
        COUNT(DISTINCT CASE WHEN data_value IS NOT NULL THEN state_name END) AS has_data,
        COUNT(DISTINCT CASE WHEN data_value IS NULL THEN state_name END) AS missing_data
    FROM stg_cdc_overdose
    WHERE year = 2020 AND month = 'December'
      AND indicator = 'Opioids (T40.0-T40.4,T40.6)'
      AND state_name != 'United States'
""", engine)
display(summary)

# Which states HAVE data?
has_data = pd.read_sql("""
    SELECT DISTINCT state_name, data_value
    FROM stg_cdc_overdose
    WHERE year = 2020 AND month = 'December'
      AND indicator = 'Opioids (T40.0-T40.4,T40.6)'
      AND data_value IS NOT NULL
      AND state_name != 'United States'
    ORDER BY state_name
""", engine)
print(f'\nStates with data: {len(has_data)}')
display(has_data)

,total_states,has_data,missing_data
0,51,20,31



States with data: 20


,state_name,data_value
0,Alaska,102.00
1,Colorado,987.00
2,Delaware,411.00
3,District of Columbia,409.00
4,Hawaii,73.00
5,Iowa,213.00
6,Kansas,252.00
7,Maine,415.00
8,Mississippi,369.00
9,Montana,85.00


In [42]:
# Which indicator has the MOST states with data in December 2020?
indicator_coverage = pd.read_sql("""
    SELECT indicator,
           COUNT(DISTINCT CASE WHEN data_value IS NOT NULL THEN state_name END) AS states_with_data,
           COUNT(DISTINCT state_name) AS total_states
    FROM stg_cdc_overdose
    WHERE year = 2020 AND month = 'December'
      AND state_name != 'United States'
    GROUP BY indicator
    ORDER BY states_with_data DESC
""", engine)
display(indicator_coverage)

,indicator,states_with_data,total_states
0,Percent with drugs specified,53,53
1,"Natural & semi-synthetic opioids, incl. methad...",42,51
2,Natural & semi-synthetic opioids (T40.2),42,51
3,Heroin (T40.1),38,51
4,Psychostimulants with abuse potential (T43.6),38,51
5,Methadone (T40.3),38,51
6,Cocaine (T40.5),36,51
7,Number of Drug Overdose Deaths,25,53
8,"Synthetic opioids, excl. methadone (T40.4)",22,51
9,"Natural, semi-synthetic, & synthetic opioids, ...",21,51


In [43]:
# Check the broadest death count indicator
broad = pd.read_sql("""
    SELECT state_name, data_value
    FROM stg_cdc_overdose
    WHERE year = 2020 AND month = 'December'
      AND indicator = 'Number of Drug Overdose Deaths'
      AND state_name != 'United States'
      AND data_value IS NOT NULL
    ORDER BY data_value DESC
    LIMIT 15
""", engine)
print(f'States with "Number of Drug Overdose Deaths": {len(broad)}')
display(broad)

States with "Number of Drug Overdose Deaths": 15


,state_name,data_value
0,Alabama,989.00
1,Nevada,915.00
2,Oregon,797.00
3,New Mexico,792.00
4,Oklahoma,746.00
5,Utah,622.00
6,Mississippi,532.00
7,Arkansas,514.00
8,District of Columbia,511.00
9,Maine,490.00


In [44]:
# Check: what about using "Number of Drug Overdose Deaths" 
# (the TOTAL, not opioid-specific) — this has 25 states
# But can we use a DIFFERENT indicator with more coverage?

# Let's try the broadest opioid-related indicator
best_indicator = pd.read_sql("""
    SELECT state_name, data_value, predicted_value
    FROM stg_cdc_overdose
    WHERE year = 2020 AND month = 'December'
      AND indicator LIKE 'Natural & semi-synthetic opioids, incl. methad%%'
      AND state_name != 'United States'
      AND data_value IS NOT NULL
    ORDER BY data_value DESC
""", engine)
print(f'States with data: {len(best_indicator)}')
display(best_indicator.head(15))
display(best_indicator.tail(10))

States with data: 42


,state_name,data_value,predicted_value
0,Illinois,696.00,706.00
1,New York City,669.00,670.00
2,Texas,641.00,647.00
3,New York,619.00,659.00
4,Tennessee,617.00,626.00
5,Maryland,601.00,601.00
6,Ohio,512.00,512.00
7,Michigan,494.00,499.00
8,Georgia,486.00,487.00
9,Kentucky,470.00,470.00


,state_name,data_value,predicted_value
32,Kansas,78.00,78.00
33,District of Columbia,66.00,66.00
34,Iowa,66.00,66.00
35,New Hampshire,44.00,44.00
36,Alaska,44.00,44.00
37,Montana,39.00,39.00
38,Vermont,38.00,38.00
39,Wyoming,30.00,30.00
40,Hawaii,29.00,29.00
41,South Dakota,11.00,11.00


In [45]:
# Can predicted_value fill the gaps?
predicted_fill = pd.read_sql("""
    SELECT state_name, data_value, predicted_value
    FROM stg_cdc_overdose
    WHERE year = 2020 AND month = 'December'
      AND indicator = 'Opioids (T40.0-T40.4,T40.6)'
      AND state_name != 'United States'
      AND data_value IS NULL
      AND predicted_value IS NOT NULL
    ORDER BY state_name
""", engine)
print(f'States where predicted_value fills the gap: {len(predicted_fill)}')
if len(predicted_fill) > 0:
    display(predicted_fill.head(10))

States where predicted_value fills the gap: 0


In [46]:
# What if we use "Number of Drug Overdose Deaths" — does it cover more states?
# AND combine with the opioid-specific indicator where available?
all_deaths = pd.read_sql("""
    SELECT state_name, indicator, data_value
    FROM stg_cdc_overdose
    WHERE year = 2020 AND month = 'December'
      AND state_name != 'United States'
      AND data_value IS NOT NULL
      AND indicator IN (
          'Number of Drug Overdose Deaths',
          'Opioids (T40.0-T40.4,T40.6)',
          'Natural & semi-synthetic opioids, incl. methadone (T40.2, T40.3)',
          'Natural & semi-synthetic opioids (T40.2)'
      )
    ORDER BY state_name, indicator
""", engine)
# Pivot to see which states have which indicators
pivot = all_deaths.pivot_table(index='state_name', columns='indicator', values='data_value')
print(f'States with ANY death data: {len(pivot)}')
display(pivot)

States with ANY death data: 48


indicator,Natural & semi-synthetic opioids (T40.2),"Natural & semi-synthetic opioids, incl. methadone (T40.2, T40.3)",Number of Drug Overdose Deaths,"Opioids (T40.0-T40.4,T40.6)"
state_name,,,,
Alabama,NaN,NaN,989.00,NaN
Alaska,37.00,44.00,146.00,102.00
Arizona,318.00,386.00,NaN,NaN
Arkansas,NaN,NaN,514.00,NaN
Colorado,270.00,330.00,NaN,987.00
Connecticut,217.00,311.00,NaN,NaN
Delaware,50.00,82.00,449.00,411.00
District of Columbia,38.00,66.00,511.00,409.00
Georgia,439.00,486.00,NaN,NaN


### ✏️ CDC Profiling Findings

**1. Total rows**: 163,800 rows covering 54 distinct state entries, years 2015–2025.

**2. Target indicator**: Our planned indicator `Opioids (T40.0-T40.4,T40.6)` only has data for **20 of 51 states** — insufficient for state-level analysis.

**⚠️ Critical discovery**: CDC death data is fragmented across indicators. No single indicator covers all states. Coverage by indicator (December 2020):

| Indicator | States with Data |
|-----------|-----------------|
| Natural & semi-synthetic opioids, incl. methadone (T40.2, T40.3) | 42 |
| Natural & semi-synthetic opioids (T40.2) | 42 |
| Heroin (T40.1) | 38 |
| Psychostimulants with abuse potential (T43.6) | 38 |
| Methadone (T40.3) | 38 |
| Cocaine (T40.5) | 36 |
| Number of Drug Overdose Deaths | 25 |
| Opioids (T40.0-T40.4,T40.6) | 20 |

**Decision**: Use `Natural & semi-synthetic opioids, incl. methadone (T40.2, T40.3)` as the primary opioid death indicator (42 states). For the remaining 9 states missing opioid-specific data, fall back to `Number of Drug Overdose Deaths` as a proxy. Document this as a limitation — the proxy overstates opioid deaths since it includes all drug overdoses.

**3. "United States" aggregate row**: Present — must exclude from state-level analysis.

**4. "New York City" appears as a separate entry** from "New York" — must combine or exclude in Phase 3.

**5. Period type**: All rows are "12 month-ending" — confirmed. December 2020 row represents deaths from Jan–Dec 2020 (full calendar year). ✅

**6. 2020 completeness**: All states show 100% completeness for December 2020. No states below the 85% threshold. ✅

**7. 2020 pending investigation**: 0 states with >15% pending investigation. Maximum pending was 0.40% (California). ✅

**8. Duplicate rows**: Every row in the CDC file appears twice (identical duplicates). Must deduplicate in Phase 3 — likely caused by the CSV export.

**Decision**: Deduplicate by keeping `DROP DUPLICATES` on all columns during Phase 3 cleaning.

**9. Data Value vs Predicted Value**: Correlation = 1.00, mean absolute difference = 3.70 for 2020. Data Value is highly reliable for finalized years. ✅

**10. Indicator consistency**: All T40 opioid indicators are present across all 11 years (2015–2025). Indicator naming is stable. ✅

**Cleaning needed for Phase 3**:
- [ ] Deduplicate all rows (every row appears twice)
- [ ] Exclude "United States" aggregate row
- [ ] Exclude "New York City" (already counted within "New York") or investigate if NYC needs to be combined with NY state
- [ ] Exclude territories (Puerto Rico, etc.)
- [ ] Use indicator hierarchy: primary = "Natural & semi-synthetic opioids, incl. methadone (T40.2, T40.3)", fallback = "Number of Drug Overdose Deaths"
- [ ] Join on `state_name` column (not `state`) since `state` has abbreviations while `dim_state` expects full names via `state_name`

### 3.3 AHRQ SDOH Profiling (7 Checks)

> Run these AFTER completing SDOH column mapping in Section 2.3.

In [47]:
# Shape
display(pd.read_sql('SELECT COUNT(*) AS rows, COUNT(DISTINCT statefips) AS states, COUNT(DISTINCT countyfips) AS counties FROM stg_ahrq_sdoh', engine))

,rows,states,counties
0,3234,56,3234


In [48]:
# 3.3.2 — Missing values
display(pd.read_sql("""
    SELECT COUNT(*) AS total,
        SUM(CASE WHEN pct_poverty IS NULL THEN 1 ELSE 0 END) AS null_poverty,
        SUM(CASE WHEN pct_unemployed IS NULL THEN 1 ELSE 0 END) AS null_unemploy,
        SUM(CASE WHEN median_hh_income IS NULL THEN 1 ELSE 0 END) AS null_income,
        SUM(CASE WHEN gini_index IS NULL THEN 1 ELSE 0 END) AS null_gini,
        SUM(CASE WHEN pct_uninsured IS NULL THEN 1 ELSE 0 END) AS null_uninsured,
        SUM(CASE WHEN total_population IS NULL OR total_population=0 THEN 1 ELSE 0 END) AS null_zero_pop
    FROM stg_ahrq_sdoh
""", engine).T.rename(columns={0:'count'}))

,count
total,3234
null_poverty,13
null_unemploy,13
null_income,14
null_gini,13
null_uninsured,13
null_zero_pop,13


In [49]:
# Summary statistics
display(pd.read_sql("""
    SELECT ROUND(AVG(pct_poverty)::numeric,2) AS avg_poverty, ROUND(MAX(pct_poverty)::numeric,2) AS max_poverty,
           ROUND(AVG(pct_unemployed)::numeric,2) AS avg_unemploy, ROUND(AVG(median_hh_income)::numeric,0) AS avg_income,
           ROUND(AVG(gini_index)::numeric,4) AS avg_gini, ROUND(AVG(pct_uninsured)::numeric,2) AS avg_uninsured,
           ROUND(SUM(total_population)/1e6,1) AS pop_millions
    FROM stg_ahrq_sdoh WHERE total_population > 0
""", engine).T.rename(columns={0:'value'}))
print('Total pop should be ~328-332M (2020 Census).')

,value
avg_poverty,15.38
max_poverty,66.19
avg_unemploy,5.45
avg_income,"54,172.00"
avg_gini,0.45
avg_uninsured,9.46
pop_millions,329.80


Total pop should be ~328-332M (2020 Census).


In [50]:
# State FIPS join check
sdoh_jn = pd.read_sql("""
    SELECT s.statefips, d.state_abbrev, COUNT(*) AS counties
    FROM stg_ahrq_sdoh s LEFT JOIN dim_state d ON s.statefips=d.state_fips
    GROUP BY s.statefips, d.state_abbrev ORDER BY d.state_abbrev NULLS LAST
""", engine)
um = sdoh_jn[sdoh_jn['state_abbrev'].isna()]
print(f'Matched: {len(sdoh_jn)-len(um)}, Unmatched: {len(um)}')
if len(um)>0: display(um)

Matched: 51, Unmatched: 5


,statefips,state_abbrev,counties
51,60,NaN,5
52,69,NaN,3
53,66,NaN,1
54,72,NaN,78
55,78,NaN,3


In [51]:
# 3.3.5 — FIPS validation + MN check
display(pd.read_sql('SELECT LENGTH(countyfips) AS len, COUNT(*) AS rows FROM stg_ahrq_sdoh GROUP BY LENGTH(countyfips)', engine))
mn = pd.read_sql("SELECT countyfips, county FROM stg_ahrq_sdoh WHERE statefips='27' AND countyfips IN ('27111','27165')", engine)
print(f'MN FIPS issue: {"found" if len(mn)>0 else "not present"}')
if len(mn)>0: display(mn)

,len,rows
0,5,3234


MN FIPS issue: found


,countyfips,county
0,27111,Todd County
1,27165,Washington County


In [52]:
# Outliers
display(pd.read_sql("""
    (SELECT 'HIGH' AS end, county, state, pct_poverty, total_population FROM stg_ahrq_sdoh WHERE pct_poverty IS NOT NULL ORDER BY pct_poverty DESC LIMIT 5)
    UNION ALL
    (SELECT 'LOW', county, state, pct_poverty, total_population FROM stg_ahrq_sdoh WHERE pct_poverty IS NOT NULL ORDER BY pct_poverty ASC LIMIT 5)
""", engine))

,end,county,state,pct_poverty,total_population
0,HIGH,Guánica Municipio,Puerto Rico,66.19,15825
1,HIGH,Adjuntas Municipio,Puerto Rico,64.78,17614
2,HIGH,Lajas Municipio,Puerto Rico,61.36,22391
3,HIGH,Arroyo Municipio,Puerto Rico,60.09,17502
4,HIGH,Todd County,South Dakota,58.87,10308
5,LOW,Loving County,Texas,0.00,117
6,LOW,Kalawao County,Hawaii,1.16,436
7,LOW,Morgan County,Utah,1.69,11950
8,LOW,Falls Church city,Virginia,2.00,14309
9,LOW,Kenedy County,Texas,2.05,391


In [53]:
# Population validity
display(pd.read_sql("""
    SELECT COUNT(*) AS counties, SUM(CASE WHEN total_population IS NULL THEN 1 ELSE 0 END) AS null_pop,
           SUM(CASE WHEN total_population=0 THEN 1 ELSE 0 END) AS zero_pop,
           ROUND(SUM(total_population)/1e6,1) AS pop_millions
    FROM stg_ahrq_sdoh
""", engine).T.rename(columns={0:'value'}))

,value
counties,"3,234.00"
null_pop,13.00
zero_pop,0.00
pop_millions,329.80


### ✏️ SDOH Findings
_(Fill in: county count, missing rates, population total, FIPS issues, outliers)_

---
## 4. Cross-Dataset Join Validation

In [54]:
# CMS -> dim_state
j = pd.read_sql("SELECT c.st, CASE WHEN d.state_abbrev IS NULL THEN 'NO MATCH' ELSE 'OK' END AS status FROM (SELECT DISTINCT TRIM(prscrbr_state_abrvtn) AS st FROM stg_cms_partd) c LEFT JOIN dim_state d ON c.st=d.state_abbrev ORDER BY status DESC, c.st", engine)
nm = j[j['status']=='NO MATCH']
print(f'CMS->dim_state: {len(j)-len(nm)} matched, {len(nm)} unmatched')
if len(nm)>0: display(nm)

CMS->dim_state: 51 matched, 9 unmatched


,st,status
51,AA,NO MATCH
52,AE,NO MATCH
53,AP,NO MATCH
54,GU,NO MATCH
55,MP,NO MATCH
56,PR,NO MATCH
57,VI,NO MATCH
58,XX,NO MATCH
59,ZZ,NO MATCH


In [55]:
# CDC -> dim_state
j = pd.read_sql("SELECT c.state, CASE WHEN d.state_abbrev IS NULL THEN 'NO MATCH' ELSE 'OK' END AS status FROM (SELECT DISTINCT state FROM stg_cdc_overdose) c LEFT JOIN dim_state d ON c.state=d.state_name ORDER BY status DESC, c.state", engine)
nm = j[j['status']=='NO MATCH']
print(f'CDC->dim_state: {len(j)-len(nm)} matched, {len(nm)} unmatched')
if len(nm)>0: display(nm)

CDC->dim_state: 0 matched, 54 unmatched


,state,status
0,AK,NO MATCH
1,AL,NO MATCH
2,AR,NO MATCH
3,AZ,NO MATCH
4,CA,NO MATCH
5,CO,NO MATCH
6,CT,NO MATCH
7,DC,NO MATCH
8,DE,NO MATCH
9,FL,NO MATCH


In [56]:
# AHRQ -> dim_state
j = pd.read_sql("SELECT s.statefips, CASE WHEN d.state_abbrev IS NULL THEN 'NO MATCH' ELSE 'OK' END AS status FROM (SELECT DISTINCT statefips FROM stg_ahrq_sdoh) s LEFT JOIN dim_state d ON s.statefips=d.state_fips ORDER BY status DESC", engine)
nm = j[j['status']=='NO MATCH']
print(f'AHRQ->dim_state: {len(j)-len(nm)} matched, {len(nm)} unmatched')
if len(nm)>0: display(nm)

AHRQ->dim_state: 51 matched, 5 unmatched


,statefips,status
51,72,NO MATCH
52,60,NO MATCH
53,69,NO MATCH
54,78,NO MATCH
55,66,NO MATCH


In [57]:
# CDC join — use state_name column instead of state
j = pd.read_sql("""
    SELECT c.state_name, 
           CASE WHEN d.state_abbrev IS NULL THEN 'NO MATCH' ELSE 'OK' END AS status
    FROM (SELECT DISTINCT state_name FROM stg_cdc_overdose) c
    LEFT JOIN dim_state d ON c.state_name = d.state_name
    ORDER BY status DESC, c.state_name
""", engine)
nm = j[j['status'] == 'NO MATCH']
print(f'CDC (via state_name) -> dim_state: {len(j)-len(nm)} matched, {len(nm)} unmatched')
if len(nm) > 0:
    display(nm) 

CDC (via state_name) -> dim_state: 51 matched, 3 unmatched


,state_name,status
51,New York City,NO MATCH
52,Puerto Rico,NO MATCH
53,United States,NO MATCH


### ✏️ Cross-Dataset Join Validation Summary

| Join | Key Column | Matched | Unmatched | Action |
|------|-----------|---------|-----------|--------|
| CMS → dim_state | `prscrbr_state_abrvtn` = `state_abbrev` | 51 | 9 (AA, AE, AP, GU, MP, PR, VI, XX, ZZ) | Exclude territories/military |
| CDC → dim_state | `state_name` = `state_name` | 51 | 3 (United States, New York City, Puerto Rico) | Exclude aggregate + NYC + territory |
| AHRQ → dim_state | `statefips` = `state_fips` | 51 | 5 (FIPS 60, 66, 69, 72, 78) | Exclude territories |

**Key finding**: The CDC `state` column contains abbreviations, NOT full names. The join must use the `state_name` column instead. This will be implemented in the Phase 3 master merge query.

**All 51 U.S. jurisdictions (50 states + DC) are joinable across all three datasets.** ✅

**Analysis year**: 2020 (aligned across CMS, CDC, and AHRQ)  
**Analysis grain**: State level (51 rows in final fact table)

---
## 5. Checkpoint

**Completed**: Schema + data loading + profiling (27 checks) + join validation  
**Next (Part 2)**: 7 EDA visualizations + quality summary + 10 documented decisions